# 09 딥러닝 예측 모델 (Phase 1 — 3/3)

**40조건** × **LSTM, Autoformer, N-HiTS, iTransformer**

07·08 결과와 합쳐 **10모델 중 조건별 Best** 선정

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


Torch device: cpu
시계열: 165
학습 <= 201730 | 검증: [201731, 201732, 201733]
Best 선정 기준: MAPE


### ① DL 모델 실험

In [2]:
# Phase 1 — DL 4종: SBC 20조건 + ML 20조건
cache = DATA_PROCESSED / 'phase1_dl_results.parquet'
if cache.exists():
    results = pd.read_parquet(cache)
    summary, best = summarize_phase1(results)
    print('캐시 로드 |', len(results), 'rows')
else:
    sbc = run_phase1_all(df, feat_df, 'SBC_CLUSTER', 'SBC', models=DL_MODELS)
    ml = run_phase1_all(df, feat_df, 'ML_CLUSTER', 'ML', models=DL_MODELS)
    results = pd.concat([sbc, ml], ignore_index=True)
    summary, best = summarize_phase1(results)
    results.to_parquet(cache, index=False)
    summary.to_csv(DATA_PROCESSED / 'phase1_dl_results_summary.csv', index=False)
    best.to_csv(DATA_PROCESSED / 'phase1_dl_results_best.csv', index=False)
    print('완료 |', len(results), 'rows')

display(best.sort_values(['cluster_scheme', 'type', 'cluster']))
print('\n=== 알고리즘별 평균', RANK_METRIC.upper(), '===')
print(results.groupby(['cluster_scheme', 'model'])[RANK_METRIC].mean().unstack('cluster_scheme').round(2))


Phase1 SBC:   0%|          | 0/20 [00:00<?, ?it/s]

Phase1 ML:   0%|          | 0/20 [00:00<?, ?it/s]

완료 | 1320 rows


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
2,ML,A,1,N-HiTS,3180.003175,4439.369833,124.475991,4.766104
4,ML,A,2,Autoformer,352122.899811,393388.217849,99.981276,9.143719
8,ML,A,3,Autoformer,249677.182670,276946.735557,99.974979,10.868138
12,ML,A,4,Autoformer,63007.308419,70285.350638,99.884501,10.076318
19,ML,B,1,iTransformer,7143.572847,7830.186255,95.724644,7.114177
22,ML,B,4,N-HiTS,56874.305819,78328.569487,96.219463,3.444235
27,ML,C,1,iTransformer,4937.281100,5438.022796,93.560792,7.099927
30,ML,C,2,N-HiTS,94981.291667,136586.241830,89.994409,4.094335
34,ML,C,4,N-HiTS,39233.938921,50704.569007,80.043413,4.798019
37,ML,D,1,LSTM,6697.581146,7448.253709,113.888868,7.985348



=== 알고리즘별 평균 MAPE ===
cluster_scheme      ML     SBC
model                         
Autoformer      119.91  116.07
LSTM            107.60  103.88
N-HiTS          131.38  130.55
iTransformer    115.72  111.10


### ② Phase 1 통합 Best (10모델)

In [3]:
stat = pd.read_parquet(DATA_PROCESSED / 'phase1_stat_results.parquet')
ml = pd.read_parquet(DATA_PROCESSED / 'phase1_ml_results.parquet')
dl = pd.read_parquet(DATA_PROCESSED / 'phase1_dl_results.parquet')

phase1_all = pd.concat([stat, ml, dl], ignore_index=True)
phase1_summary, phase1_best = merge_phase1_best(stat, ml, dl)
phase1_all.to_parquet(DATA_PROCESSED / 'phase1_all_results.parquet', index=False)
phase1_summary.to_csv(DATA_PROCESSED / 'phase1_summary.csv', index=False)
phase1_best.to_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv', index=False)
print('Phase1 통합 Best |', len(phase1_best), '조건')
display(phase1_best.sort_values(['cluster_scheme', 'type', 'cluster']))


Phase1 통합 Best | 35 조건


,cluster_scheme,type,cluster,best_model,mae_mean,rmse_mean,best_mape,mase_mean
8,ML,A,1,XGBoost,2462.732860,3361.175379,99.807025,4.524735
11,ML,A,2,Autoformer,352122.899811,393388.217849,99.981276,9.143719
28,ML,A,3,XGBoost,58266.037150,80798.303933,59.800775,2.536248
38,ML,A,4,XGBoost,22253.233989,32108.340274,96.838858,3.578794
49,ML,B,1,iTransformer,7143.572847,7830.186255,95.724644,7.114177
58,ML,B,4,XGBoost,40878.801764,58713.693148,75.042206,2.771052
69,ML,C,1,iTransformer,4937.281100,5438.022796,93.560792,7.099927
75,ML,C,2,RF,75887.663677,113124.954524,74.104331,3.271271
86,ML,C,4,SBA,41325.891221,49425.655750,76.696644,4.954215
92,ML,D,1,LSTM,6697.581146,7448.253709,113.888868,7.985348
